In [2]:
import pandas as pd
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, matthews_corrcoef

# 1. Load Data
print("Loading train.csv...")
df = pd.read_csv('train.csv')

# 2. Split Data (80% Training, 20% Testing)
X = df.drop(columns=['price_range'])
y = df['price_range']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 3. Save test data for the Streamlit app
test_df = pd.concat([X_test, y_test], axis=1)
test_df.to_csv('test_data.csv', index=False)
print("Saved 'test_data.csv' (400 samples) for Streamlit upload.\n")

# 4. Create Model Directory
os.makedirs('model', exist_ok=True)

# 5. Define Models 
# (Using Pipeline to standard scale data automatically for Logistic Regression)
models = {
    "Logistic Regression": Pipeline([
        ('scaler', StandardScaler()), 
        ('lr', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "KNN": KNeighborsClassifier(), 
    "Naive Bayes": GaussianNB(),
    "Random Forest": RandomForestClassifier(random_state=42)
}

# 6. Train, Evaluate, and Save Models
results = {}
print("Training models, saving them, and calculating metrics...")

for name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)
    
    # Save the model
    # Convert name to file-friendly format (e.g., 'Logistic Regression' -> 'logistic_regression.joblib')
    file_name = name.lower().replace(" ", "_")
    joblib.dump(model, f"model/{file_name}.joblib")
    
    # Generate predictions for evaluation
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)
    
    # Calculate metrics
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob, multi_class='ovr')
    prec = precision_score(y_test, y_pred, average='macro')
    rec = recall_score(y_test, y_pred, average='macro')
    f1 = f1_score(y_test, y_pred, average='macro')
    mcc = matthews_corrcoef(y_test, y_pred)
    
    # Store results
    results[name] = {
        'Accuracy': round(acc, 4),
        'AUC': round(auc, 4),
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1': round(f1, 4),
        'MCC': round(mcc, 4)
    }

print("\n--- Model Training Complete ---")
print("All models have been saved to the 'model/' directory.\n")

# 7. Print the Evaluation Metrics Table (For your README)
print("Evaluation Metrics Comparison:")
results_df = pd.DataFrame(results).T
print(results_df.to_markdown())

Loading train.csv...
Saved 'test_data.csv' (400 samples) for Streamlit upload.

Training models, saving them, and calculating metrics...

--- Model Training Complete ---
All models have been saved to the 'model/' directory.

Evaluation Metrics Comparison:
|                     |   Accuracy |    AUC |   Precision |   Recall |     F1 |    MCC |
|:--------------------|-----------:|-------:|------------:|---------:|-------:|-------:|
| Logistic Regression |      0.965 | 0.9987 |      0.965  |    0.965 | 0.965  | 0.9534 |
| Decision Tree       |      0.83  | 0.8867 |      0.8319 |    0.83  | 0.8302 | 0.7738 |
| KNN                 |      0.935 | 0.9914 |      0.9346 |    0.935 | 0.9347 | 0.9134 |
| Naive Bayes         |      0.81  | 0.9506 |      0.8113 |    0.81  | 0.8105 | 0.7468 |
| Random Forest       |      0.88  | 0.9769 |      0.8796 |    0.88  | 0.8797 | 0.84   |
